# Threads & concurrency · **Python side**

*Python / Colab track — analogue of the Julia notebook `3_threads_and_concurrency.jl`.*

Two ideas, deliberately kept apart — exactly like the Julia notebook:

- **Part A — concurrency on one core.** Overlap the **waiting** (network). Tools: `asyncio` / threads.
- **Part B — parallelism on many cores.** Real CPU work — where the **GIL** finally bites, and where
  the **race condition** lives.

> The single most misunderstood point about the GIL is settled in Part A: **it does *not* stop threads
> from overlapping I/O.** It only stops them running Python *bytecode* at the same time.

> On Colab: `Runtime ▸ Run all`. Part A needs internet (Binance public API); Part B uses `fork`
> (Linux) so `ProcessPool` works — functions are defined at module level.

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Time per call (s), %timeit-style: auto-scale then min over `repeat` trials."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"

# Part A — concurrency: overlapping I/O on one core

Fetching 15 prices one after another means **waiting** 15 times. The CPU does nothing during those
waits. **Concurrency** = fill the waiting, not compute faster.

We fetch the **same Binance spot prices as the Julia notebook**, so the numbers are directly
comparable in class.

In [ ]:
import asyncio, time, re, urllib.request
from concurrent.futures import ThreadPoolExecutor

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
           "ADAUSDT", "DOGEUSDT", "TRXUSDT", "DOTUSDT", "LTCUSDT",
           "AVAXUSDT", "LINKUSDT", "ATOMUSDT", "UNIUSDT", "ETCUSDT"]

def fetch_price(sym):
    """One blocking HTTP GET (stdlib only — no pip install)."""
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={sym}"
    with urllib.request.urlopen(url, timeout=20) as r:
        return float(re.search(r'"price":"([0-9.]+)"', r.read().decode()).group(1))

print("BTCUSDT =", fetch_price("BTCUSDT"), "USDT")

In [ ]:
# We time each ONCE (never %timeit here: it would hammer the API).

# 1) SEQUENTIAL — request, wait, next... = the SUM of the latencies
t0 = time.perf_counter()
seq = [fetch_price(s) for s in SYMBOLS]
t_seq = time.perf_counter() - t0
print(f"1) sequential             : {t_seq:7.3f} s")

# 2) THE WRONG BENCHMARK — build the coroutines but never await them.
STARTED = []                                         # a coroutine body appends here when it RUNS

async def fetch_async(s):
    STARTED.append(s)                                # ...does this ever happen?
    return await asyncio.to_thread(fetch_price, s)   # run the blocking call off the event loop

#    Note we do this INSIDE a running event loop, so nobody can object
#    "well, of course nothing ran — there was no loop".
async def bad_bench():
    t0 = time.perf_counter()
    coros = [fetch_async(s) for s in SYMBOLS]        # nothing happens!
    t_bad = time.perf_counter() - t0
    print(f"2) coroutines NOT awaited : {t_bad:9.6f} s   <- only their CREATION was timed!")
    print(f"   bodies actually started: {len(STARTED)}/{len(SYMBOLS)}")
    await asyncio.sleep(2)                           # the loop is running, idle, waiting
    print(f"   ...2 s later, loop idle: {len(STARTED)}/{len(SYMBOLS)}   <- they NEVER start. Compare with Julia!")
    for c in coros:
        c.close()                                    # avoid "never awaited" warnings
    return t_bad

t_bad = asyncio.run(bad_bench())
STARTED.clear()

# 3) THE REAL THING — asyncio.gather IS Julia's @sync + @async in one call
async def main():
    return await asyncio.gather(*[fetch_async(s) for s in SYMBOLS])

t0 = time.perf_counter()
good = asyncio.run(main())
t_good = time.perf_counter() - t0
print(f"3) asyncio.gather         : {t_good:7.3f} s   <- ~one round-trip, speedup {t_seq/t_good:.1f}x")
print(f"   bodies started         : {len(STARTED)}/{len(SYMBOLS)}   <- awaiting is what starts them")

# 4) AND PLAIN THREADS? The GIL is supposed to ruin everything... let's see.
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(SYMBOLS)) as ex:
    thr = list(ex.map(fetch_price, SYMBOLS))
t_thr = time.perf_counter() - t0
print(f"4) ThreadPoolExecutor     : {t_thr:7.3f} s   <- speedup {t_seq/t_thr:.1f}x  ... threads DO help!")

### What just happened

1. **Sequential** ≈ the *sum* of 15 latencies.
2. **Coroutines never awaited** ≈ a few µs — and **`0/15` bodies ever ran**, even 2 seconds later.
   A coroutine is **lazy**: `fetch_async(s)` builds an object and executes *nothing*. You timed
   object creation. Python would even warn you *"coroutine was never awaited"*.
3. **`asyncio.gather`** ≈ **one** round-trip, and now `15/15` bodies ran. `gather` **is** `@sync` +
   `@async` in a single call: fire everything, then wait for all of it.
4. **Plain threads also work** — and this is the point everyone gets wrong.

> ⚠️ **The GIL does *not* make threads useless for I/O.** CPython **releases** the GIL while a thread
> waits on the network, so the waits genuinely overlap. The GIL only forbids running Python
> **bytecode** simultaneously — it kills **CPU parallelism** (Part B), not **I/O concurrency**.

*(Note: `asyncio.to_thread` runs the blocking call in a thread — legitimate in Python precisely because
the GIL is released on I/O. A production async client such as `aiohttp` would do true single-threaded
async I/O, at the price of a `pip install`.)*

### 🔁 The same trap, the opposite cause — and *that* is the lesson

Julia's notebook runs the identical experiment and gets the identical **shape**: sequential ≈ 4 s,
"forgot to synchronise" ≈ 10 µs, real version ≈ 0.3 s. It is tempting to call these the same bug.
**They are not.** Look at what the two languages do while you aren't watching:

| | Julia: `@async` without `@sync` | Python: coroutine never awaited |
|---|---|---|
| when you write it | the task is **scheduled immediately** | an object is **built**, nothing runs |
| the requests | **really in flight** | **never sent** |
| 1 second later | **15/15 done** — the data is there | **0/15** — the bodies never started |
| what you measured | task creation (you looked too early) | object creation (there was nothing to look at) |
| the actual bug | you didn't **wait** for the answer | you didn't **ask the question** |

> **Julia forgot to wait; Python forgot to start.**

Both give you a meaningless benchmark and (in Julia's case) possibly garbage data, so the *discipline*
is the same: **the thing that waits is also the thing that makes the measurement real**. But the
mechanisms are opposites — `@async` is eager, `async def` is lazy. Confusing them is how you write a
Python program that silently does nothing, or a Julia program that races.

# Part B — parallelism: the GIL bites on CPU

Part A was about *waiting*. Now we actually **compute** — and everything changes.

## 1. The GIL: `threading` does not speed up CPU work

CPython runs **only one thread of bytecode at a time** (Global Interpreter Lock). On pure CPU work the
threads take turns and never truly run at once. **Processes** each own an interpreter → real
parallelism (at the price of copying data).

In [ ]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor

# CPU-bound pure-Python function (deliberately not vectorised)
def cpu_load(rep):
    s = 0.0
    for i in range(rep):
        s += (i % 7) * 1.0000001
    return s
ITER = 5_000_000

In [ ]:
def run(executor_cls, workers, tasks):
    with executor_cls(max_workers=workers) as ex:
        list(ex.map(cpu_load, tasks))

tasks = [ITER]*8

t_seq_cpu = best_time(lambda: [cpu_load(t) for t in tasks], repeat=3)
t_thr_cpu = best_time(lambda: run(ThreadPoolExecutor, 8, tasks), repeat=3)
t_prc_cpu = best_time(lambda: run(ProcessPoolExecutor, 8, tasks), repeat=3)

print("sequential      :", ms(t_seq_cpu))
print("8 threads (GIL) :", ms(t_thr_cpu), f"-> speedup {t_seq_cpu/t_thr_cpu:.2f}x  (~1: no gain at all)")
print("8 processes     :", ms(t_prc_cpu), f"-> speedup {t_seq_cpu/t_prc_cpu:.2f}x  (real parallelism)")

**Verdict.** Threads ≈ sequential (the GIL serialises them); processes scale ~× cores.

> **The same threads that gave us a ~12× speedup in Part A give ~1× here.** Nothing changed about the
> GIL — what changed is that we're now *computing* instead of *waiting*. That contrast **is** the
> difference between concurrency and parallelism.

> Julia has **no** GIL: `Threads.@spawn` parallelises for real, in shared memory. More powerful… and
> that's exactly what opens the door to the *race condition* below.

## 2. The race condition

`counter += 1` is **not** atomic: it is read → add → write back, across several bytecodes.

An honest nuance about the GIL: on CPython a tight loop often **finishes before any preemption**,
which *hides* the bug — but does not **fix** it. To make the interleaving visible we insert
`time.sleep(0)` between the read and the write: it **yields** to another thread at the worst moment.
That is an execution the scheduler is **always allowed** to produce — and one that the **free-threaded**
build of Python (≥ 3.13) produces with **no** `sleep` at all.

In [ ]:
import threading

counter = 0
def incr(n):
    global counter
    for _ in range(n):
        v = counter
        time.sleep(0)          # yield: exposes the interleaving (always permitted)
        counter = v + 1       # writes back a stale value -> increments are lost

K, T = 2000, 8
counter = 0
threads = [threading.Thread(target=incr, args=(K,)) for _ in range(T)]
for t in threads: t.start()
for t in threads: t.join()

print("expected :", K*T)
print("got      :", counter, " -> increments lost, and the number changes every run")

## 3. Fixing it: a lock, or separate accumulators

As in Julia, the right approach is to **not share** in the hot loop: each worker accumulates on its
own side and we combine at the end (here via processes).

In [ ]:
# correct: a lock makes the increment indivisible (but serialises -> slow)
counter = 0
lock = threading.Lock()
def incr_lock(n):
    global counter
    for _ in range(n):
        with lock:
            counter += 1

counter = 0
threads = [threading.Thread(target=incr_lock, args=(K,)) for _ in range(T)]
for t in threads: t.start()
for t in threads: t.join()
print("with a lock — expected :", K*T, " got :", counter)

## 4. The red thread: Monte-Carlo π in parallel (processes)

⚠️ **Before the code: where do the random numbers come from?** Every worker is about to draw
millions of values. A generator is a *mutable object updated on every draw* — the exact thing we
just spent a section proving you must not share. So each worker needs **its own stream**, and for a
simulation study you also want those streams to be **reproducible**.

`np.random.default_rng()` with no argument seeds itself from the OS: independent, but you can never
run it again. The right tool is **`SeedSequence.spawn`** — one master seed, `n` provably independent
child streams, reproducible:

```python
children = np.random.SeedSequence(1234).spawn(workers)   # n independent streams
rngs = [np.random.default_rng(c) for c in children]
```

*(Julia twin: `rand()` uses a `TaskLocalRNG` — every task gets its own stream **automatically**,
seeded deterministically from its parent. Python makes you ask; Julia gives it to you by default.
Same requirement, and the GPU section pushes it to millions of streams at once.)*

In [ ]:
def pi_chunk(args):
    """One worker: its OWN generator, from its OWN seed -> no sharing, and reproducible."""
    n, seed = args
    rng = np.random.default_rng(seed)
    x = rng.random(n); y = rng.random(n)
    return int(np.count_nonzero(x*x + y*y <= 1.0))

def pi_parallel(n, workers=8, master=1234):
    part = n // workers
    seeds = np.random.SeedSequence(master).spawn(workers)   # independent child streams
    with ProcessPoolExecutor(max_workers=workers) as ex:
        c = sum(ex.map(pi_chunk, [(part, s) for s in seeds]))
    return 4*c/(part*workers)

def pi_seq(n, master=1234):
    return 4*pi_chunk((n, np.random.SeedSequence(master))) / n

n = 50_000_000
print("π (reference) :", np.pi)
print("sequential  :", ms(best_time(lambda: pi_seq(n), repeat=3)))
print("processes   :", ms(best_time(lambda: pi_parallel(n), repeat=3)))

# Reproducible? Same master seed, same workers -> the SAME answer, every run, every machine.
print("\nrun 1 :", pi_parallel(4_000_000))
print("run 2 :", pi_parallel(4_000_000), " <- identical")
print("8 workers vs 4 :", pi_parallel(4_000_000, workers=8), "vs", pi_parallel(4_000_000, workers=4))
# ⚠️ Changing the worker count redistributes the streams -> a different (equally valid) answer.
#    Reproducibility is tied to (master seed, number of workers), not to the seed alone.

## Wrap-up — to compare with Julia

### When does what help? (the table to remember)

| | **I/O-bound** (waiting) | **CPU-bound** (computing) |
|---|---|---|
| **Python** | threads **work** (the GIL is released during I/O), or `asyncio.gather` | threads **useless** (the GIL) → `multiprocessing` |
| **Julia** | `@async` / `@sync` (one core) | `@spawn` / `fetch` (many cores) |

Those four cells are **why "concurrency" and "parallelism" are two different words**: the enemy isn't
the same. **Waiting is overlapped; computing must be split.**

### And the rest

| | Python | Julia |
|---|---|---|
| overlap I/O | `asyncio.gather`, or threads | `@sync` + `@async` |
| the "forgot" bug | coroutine never awaited (~10 µs) — **never starts** | `@async` without `@sync` (~10 µs) — **starts, unwaited** |
| threads on pure CPU | **no gain** (GIL) | real parallelism (`@spawn`) |
| parallelise CPU | **processes** (data copied) | threads (shared memory) |
| race condition | possible (`+=` not atomic) | possible (and silent) |
| random streams | `SeedSequence(seed).spawn(k)` — **you must ask** | `TaskLocalRNG` — **automatic**, per task |

> ⚠️ Same *symptom*, opposite *cause*: `@async` is **eager** (Julia forgot to wait), `async def` is
> **lazy** (Python forgot to start). See Part A.

> Same lesson on both sides: **minimise sharing**. Python almost forces it on you (isolated
> processes); Julia lets you share memory — powerful, but at your own risk. And the random
> generator is shared state too: give every worker its **own stream**, in either language.